# Predictive Analysis — Falcon 9 Landing Classification

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

X = pd.read_csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_3.csv")
print(X.shape)
display(X.head())

In [ ]:
# The course's dataset_part_3 is already one-hot encoded.
# Make the feature matrix numeric and use the Class column as the target.
y = X["Class"].astype(int)
features = X.drop(columns=["Class"]).select_dtypes(include=[np.number])
X_train, X_test, y_train, y_test = train_test_split(
    features, y, test_size=0.2, random_state=42, stratify=y
)
print("Train:", X_train.shape, "Test:", X_test.shape)

In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("scale", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000))
    ]),
    "Linear SVM": Pipeline([
        ("scale", StandardScaler()),
        ("model", SVC(kernel="linear"))
    ]),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "KNN": Pipeline([
        ("scale", StandardScaler()),
        ("model", KNeighborsClassifier())
    ])
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    results[name] = accuracy_score(y_test, pred)
    print(f"{name}: {results[name]*100:.2f}%")

pd.Series(results).sort_values(ascending=False).plot(kind="bar", title="Classifier Test Accuracy")
plt.ylabel("Accuracy")
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

In [ ]:
# Tune the linear SVM, the model selected in the final report.
svm_pipe = Pipeline([
    ("scale", StandardScaler()),
    ("model", SVC(kernel="linear"))
])
grid = GridSearchCV(
    svm_pipe,
    {"model__C": [0.01, 0.1, 1, 10, 100]},
    cv=5,
    scoring="accuracy"
)
grid.fit(X_train, y_train)

best_svm = grid.best_estimator_
pred = best_svm.predict(X_test)

print("Best parameters:", grid.best_params_)
print("Test accuracy:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred, digits=3))
print("Confusion matrix:\n", confusion_matrix(y_test, pred))

The final presentation reported 83.33% test accuracy (15/18 correct) for the four compared classifiers and selected a linear SVM. Because the test set is small, rerunning the notebook is the authoritative result for the exact environment used.